# Task Statement 2.5: Select and Apply Built-in Tools Effectively

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 2: Tool Design & MCP Integration
- **Domain Weight**: 20% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 4 (Developer Productivity with Claude)

### The Core Goal
To master the selection criteria and execution protocols for Claude Code's built-in developer tool suite (**`Read`**, **`Write`**, **`Edit`**, **`Bash`**, **`Grep`**, **`Glob`**), and implement the deterministic **`Edit` Fallback Recovery Protocol** when surgical string replacements encounter non-unique match collisions.

### Why This Matters in Production
In real-world software engineering, tools carry stark trade-offs in token cost, latency, and safety:
- Using `Read` to search for a function name across a 50-file repository floods the context window with 40,000 irrelevant tokens, whereas **`Grep`** returns the exact file and line number in 200 tokens.
- Using `Write` to overwrite a 600-line file to change 1 line risks accidental deletions and formatting regressions, whereas **`Edit`** performs surgical, diff-safe replacements.
- If `Edit` fails because the target string appears in multiple locations, naively repeating the same edit causes an infinite loop. Understanding the exact recovery path (expanding surrounding anchor lines, or falling back to `Read` followed by `Write`) is critical to autonomous reliability.

### Exam Importance
Task Statement 2.5 questions test tool selection instincts (e.g. choosing between Grep and Glob) and failure recovery behavior. The exam frequently asks what an agent must do when an `Edit` call fails with an "ambiguous match" or "multiple matches found" status.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Built-in Tool Ecosystem Explained
Claude Code provides six native primitives:
1. **`Grep` (Content Search)**: Searches inside file contents using regex. Used to find function definitions, imported symbols, variable references, or error strings across the repository.
2. **`Glob` (Path Search)**: Matches file paths by pattern (e.g. `tests/**/*.py`, `src/**/*.tsx`). Does *not* inspect file contents.
3. **`Read` (File Inspection)**: Reads full files or specific line ranges (using `offset` and `limit`). Used *after* `Grep` or `Glob` to inspect surrounding context.
4. **`Edit` (Surgical Replacement)**: Replaces an exact string (`old_str`) with a replacement string (`new_str`) inside an existing file, preserving the rest of the file untouched.
5. **`Write` (File Creation & Overwrite)**: Writes complete file contents to disk. Used for creating brand new files or complete rewrites when surgical editing is impossible.
6. **`Bash` (Shell Execution)**: Runs compilers, test suites (`pytest`), linters (`black`, `flake8`), and git commands.

### 2. The `Edit` Collision & Fallback Recovery Protocol
To protect codebases against unintended widespread mutations, the `Edit` tool enforces a strict safety contract:
- If `old_str` matches **exactly 1 location** in the target file $\rightarrow$ **Success** (applied cleanly).
- If `old_str` matches **0 locations** $\rightarrow$ **Fails** (`NOT_FOUND`).
- If `old_str` matches **2 or more locations** $\rightarrow$ **Fails** (`AMBIGUOUS_COLLISION`).

#### The 3-Step Recovery Hierarchy:
1. **Initial Attempt**: Attempt surgical `Edit(path, old_str, new_str)`.
2. **Step 2 (Anchor Expansion)**: If `AMBIGUOUS_COLLISION` occurs, call `Read` to inspect 3–5 lines of code above and below the target. Re-attempt `Edit` using the **expanded unique anchor string** (including surrounding context lines).
3. **Step 3 (Read + Write Fallback)**: If anchor expansion still fails (e.g. deeply duplicated repetitive code), read the full file into memory, perform in-memory string manipulation for the specific occurrence index, and write the complete file using `Write`.

---

## 🔑 Key Technical Terminology Explained

- **`Grep` vs `Glob`**: `Grep` inspects the text *inside* files; `Glob` inspects file and directory *names* on disk.
- **Surgical `Edit`**: A tool operation that modifies only a small contiguous block of text inside a file while leaving all other lines byte-for-byte identical.
- **Anchor String**: Surrounding lines of code included in `old_str` to make the target text globally unique within the file.
- **Ambiguous Collision (`AMBIGUOUS_COLLISION`)**: An error returned by `Edit` when the target string matches multiple locations in the file, preventing ambiguous replacements.
- **Read + Write Fallback**: The secondary recovery protocol where the agent reads the complete file into memory and uses `Write` to perform a full file replacement when surgical anchors cannot be disambiguated.

---

## 🏗️ Architecture: The `Edit` Fallback Protocol

```mermaid
flowchart TD
    Start["Attempt Surgical Edit(old_str, new_str)"] --> CheckStatus{"Inspect Edit Status"}
    
    CheckStatus -->|"Success (1 match)"| Done["Code Successfully Patched"]
    
    CheckStatus -->|"Collision: 2+ matches found"| ReadContext["Step 2: Read Surrounding Lines<br>(Inspect 3-5 lines above & below)"]
    CheckStatus -->|"Not Found: 0 matches"| ReadContext
    
    ReadContext --> ExpandAnchor["Construct Expanded Unique Anchor<br>(Include function header or unique line)"]
    ExpandAnchor --> ReattemptEdit["Re-attempt Edit with Expanded Anchor"]
    
    ReattemptEdit --> CheckStatus2{"Inspect Second Attempt"}
    CheckStatus2 -->|"Success"| Done
    CheckStatus2 -->|"Still Failing"| FallbackWrite["Step 3: Read + Write Fallback<br>(Read full file, replace target index, Write file)"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Scenario / Dilemma | Correct Implementation | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Searching for a Symbol** | Use **`Grep`** with symbol regex to find file and line numbers. | **TRAP**: Calling `Read` across every file in the directory tree. | Exhausts context window tokens and introduces massive unnecessary latency. |
| **Locating Test Files** | Use **`Glob`** with `tests/**/test_*.py`. | **TRAP**: Using `Grep` to search for filenames inside files. | `Grep` searches content, not directory paths; misses files without specific text strings. |
| **Fixing 1 Line in Large File** | Use surgical **`Edit`** with unique anchors. | **TRAP**: Using **`Write`** to rewrite the entire 1,000-line file. | High risk of hallucinating dropped functions, syntax regressions, and losing file permissions. |
| **Handling Edit Collision** | **Expand context anchor lines** above/below to ensure uniqueness, or fallback to `Read + Write`. | **TRAP**: Blindly re-executing the identical `Edit` call with no changes. | Fails identically on every retry until iteration limits abort the session. |

---

## 💡 Architectural Decision Matrix: Built-in Tools

| Developer Intent | Optimal Tool | Reason |
|---|---|---|
| Find which file defines `class UserAuth` | `Grep` | Content search across all files in 1 turn. |
| Find all `.json` files in `config/` | `Glob` | File path pattern matching. |
| Inspect lines 40–80 of `server.py` | `Read(offset=40, limit=40)` | Targeted window without reading full 2,000-line file. |
| Update variable name in 1 function | `Edit` | Surgical, diff-safe modification. |
| Create new test file `test_auth.py` | `Write` | New file creation from scratch. |
| Run test suite and check exit code | `Bash` | Executes `pytest` in native runtime environment. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

- **Anchor Planning**: Extended Thinking allows Claude to analyze the target file contents during its reasoning phase and construct a unique multi-line anchor string *before* calling `Edit`, avoiding collisions on the first turn.
- **Sampling Rules**: Omit `temperature` whenever thinking is enabled.
- **Thinking Continuity**: When handling collision retries, assistant thinking blocks describing the failed attempt must be preserved with their cryptographic signatures.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_2_5_builtin_tools_usage.ipynb`](ts_2_5_builtin_tools_usage.ipynb):
1. **Step 1: Setup & Developer Toolbox**: Initializes the simulated file system with Grep, Glob, Read, Write, and Edit tools.
2. **Step 2: Grep & Glob Discovery**: Demonstrates finding symbols in contents via `grep_symbols` and files via `glob_paths`.
3. **Step 3: Triggering Edit Collision**: Intentionally attempts to replace a duplicated string, triggering `AMBIGUOUS_COLLISION`.
4. **Step 4: Anchor Expansion & Fallback**: Demonstrates successful recovery using expanded multi-line context anchors.
5. **Step 5: Verification Assertions**: Verifies that 100% of duplicate collisions recover safely without corrupting unrelated code.

In [ ]:
# Step 1: Environment & Setup
import fnmatch
import os
import re
import sys
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.5 Built-in Tools Usage.")

### 🧰 Step 2: Built-in Toolbox Engine Implementation
We implement the full suite of built-in developer tools with safe fallback handling.

In [ ]:
# Step 2: BuiltinToolbox Implementation
class BuiltinToolbox:
    def __init__(self, workspace_root: Path) -> None:
        self.root = workspace_root

    def glob_files(self, pattern: str) -> List[str]:
        matches = []
        ignore_dirs = {".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache"}
        for dirpath, dirnames, filenames in os.walk(str(self.root)):
            dirnames[:] = [d for d in dirnames if d not in ignore_dirs and not d.startswith(".venv")]
            rel_dir = os.path.relpath(dirpath, str(self.root)).replace("\\", "/")
            for fname in filenames:
                rel = f"{rel_dir}/{fname}" if rel_dir != "." else fname
                if fnmatch.fnmatch(rel, pattern) or fnmatch.fnmatch(fname, pattern):
                    matches.append(rel)
        return sorted(matches)

    def grep_search(self, regex_pattern: str, file_pattern: str = "*") -> List[Dict[str, Any]]:
        results = []
        compiled = re.compile(regex_pattern)
        for rel_path in self.glob_files(file_pattern):
            full_path = self.root / rel_path
            try:
                lines = full_path.read_text(encoding="utf-8").splitlines()
                for line_no, line in enumerate(lines, start=1):
                    if compiled.search(line):
                        results.append({"file": rel_path, "line": line_no, "content": line.strip()})
            except Exception:
                continue
        return results

    def read_file(self, rel_path: str, offset: int = 1, limit: Optional[int] = None) -> Dict[str, Any]:
        full_path = self.root / rel_path
        if not full_path.exists():
            return {"success": False, "error": f"File not found: {rel_path}"}
        lines = full_path.read_text(encoding="utf-8").splitlines()
        start_idx = max(0, offset - 1)
        end_idx = len(lines) if limit is None else start_idx + limit
        slice_lines = lines[start_idx:end_idx]
        return {
            "success": True,
            "file": rel_path,
            "start_line": offset,
            "total_lines": len(lines),
            "content": "\n".join(slice_lines),
        }

    def write_file(self, rel_path: str, content: str) -> Dict[str, Any]:
        full_path = self.root / rel_path
        full_path.parent.mkdir(parents=True, exist_ok=True)
        full_path.write_text(content, encoding="utf-8")
        return {"success": True, "file": rel_path}

    def edit_file(self, rel_path: str, old_str: str, new_str: str) -> Dict[str, Any]:
        read_res = self.read_file(rel_path)
        if not read_res["success"]:
            return read_res
        content = read_res["content"]
        count = content.count(old_str)
        if count == 0:
            return {"success": False, "error": "Anchor not found."}
        if count > 1:
            return {"success": False, "error": f"Ambiguous edit: found {count} occurrences."}
        updated = content.replace(old_str, new_str, 1)
        self.write_file(rel_path, updated)
        return {"success": True, "file": rel_path}

    def execute_edit_with_fallback(self, rel_path: str, old_str: str, new_str: str, unique_context: str) -> Dict[str, Any]:
        res = self.edit_file(rel_path, old_str, new_str)
        if not res["success"]:
            # Fallback: expand anchor with surrounding unique context
            expanded_old = unique_context
            expanded_new = unique_context.replace(old_str, new_str)
            res_retry = self.edit_file(rel_path, expanded_old, expanded_new)
            if res_retry["success"]:
                return {"status": "EDIT_RETRY_SUCCESS", "details": res_retry}
            # Ultimate fallback: read full file, replace, and write
            read_full = self.read_file(rel_path)
            updated = read_full["content"].replace(old_str, new_str, 1)
            self.write_file(rel_path, updated)
            return {"status": "READ_WRITE_FALLBACK_SUCCESS"}
        return {"status": "DIRECT_EDIT_SUCCESS"}

print(" BuiltinToolbox ready.")

### 🧪 Step 3: Interactive Simulation
We create a temporary workspace, write a file with ambiguous targets, trigger the ambiguity failure, and succeed via fallback.

In [ ]:
# Step 3: Simulation
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    toolbox = BuiltinToolbox(tmp_path)

    # 1. Write file with repeated text
    sample_content = "def func_a(): return 1\ndef func_b(): return 1\n"
    toolbox.write_file("app.py", sample_content)

    # 2. Attempt ambiguous edit
    amb_res = toolbox.edit_file("app.py", "return 1", "return 2")
    print("Ambiguous Edit Attempt:", amb_res)

    # 3. Execute edit with fallback (expanding context to 'def func_a(): return 1')
    fallback_res = toolbox.execute_edit_with_fallback(
        "app.py",
        old_str="return 1",
        new_str="return 2",
        unique_context="def func_a(): return 1"
    )
    print("\nFallback Edit Execution:", fallback_res)

    # 4. Read updated content
    final_content = toolbox.read_file("app.py")
    print("\nUpdated File Content:\n", final_content["content"])

### 🔬 Step 4: Verification Assertions
We assert that:
1. Ambiguous edit fails with clear error.
2. Context expansion fallback replaces only the targeted function.
3. `func_a` returns 2, and `func_b` remains returning 1.

In [ ]:
# Step 4: Verification Checks
assert amb_res["success"] is False
assert "Ambiguous edit" in amb_res["error"]
assert fallback_res["status"] == "EDIT_RETRY_SUCCESS"
assert "def func_a(): return 2" in final_content["content"]
assert "def func_b(): return 1" in final_content["content"]

print(" All verification assertions passed! Built-in tools verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Follow the 3-step codebase discovery workflow: **Glob -> Grep -> Read (slices)**.
- **Rule 2**: Avoid dumping whole files into context when only specific functions are needed.
- **Rule 3**: When an edit fails due to ambiguity, expand the unique context lines or fall back to Read + Write.